<img src="https://upload.wikimedia.org/wikipedia/commons/9/9b/NISAR_Mission_Logo.png" width=400 align="left"/><br><br><br><br><br>



# NASA ISRO Synthetic Aperture Radar Mission
## Combined Algorithm Theoretical Basis Document and Jupyter Notebook for <br> *Classification of Wetland Inundation Extent*


Authors: Bruce Chapman, Paul Siqueira

Date: February 15, 2022

Last updated: 
- May 2026, Alexander Lewandowski and Julia White
- November 2025, Alexandra Christensen
- December 2024, Brandi Downs


### Summary
This notebook describes the ATBD for generating a wetland inundation product from NISAR time series data stacks. First, the images of the multi-temporal sequence must be well radiometrically calibrated relative to each other, to a higher precision than perhaps required through routine standard calibration of the NISAR imagery. This optional calibration step examines distributed targets that are expected to be unchanged or minimally changed in brightness over a set time span of  an image sequence. With NISAR’s 240 km swath width, it is reasonably assumed that a statistically large area, A<sub>ni</sub>, will not be inundated (or otherwise changing) during any of the 2n observations surrounding the image to be calibrated and classified. These areas will be identified through use of a priori wetlands mask and partly through image segmentation or other methods over the 2n images. <br>
A set of classes will be identified from a multitemporal average of a subset of images including:

- Inundated vegetation (presumption: dominated by double bounce scatter in HH channel) 
- Open water (presumption: low specular scattering in both channels)
- Not inundated (presumption: brighter specular scatter, volume scattering)
- Not classified (presumption: pixels do not align with the scattering model, or no data)

These classes are selected based on calibrated threshold values for the radar backscatter and other metrics. In addition, this same multi-temporal image sequence allows the algorithm to include a more sensitive change detection component for improved robustness. Change detection will allow for refinement within the multitemporal image sequence for change of class during the image sequence that may be more robust than simply classifying the image backscatter and backscatter ratio values.  


### Use Environment:
[NISAR_Inundation](../requirements.yml)

### RAM Requirements

- Demonstration cal-val sites: ~20.5GB RAM
- User defined GCOV dataset: Varies depending on dataset size

<hr>

## Step 1: Import Python Packages

In [ ]:
from collections import Counter
from datetime import datetime
from pathlib import Path
import re
from yaml import safe_load

import earthaccess
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import numpy as np
from osgeo import gdal
import pandas as pd
import rasterio
import rioxarray
import s3fs
import xarray as xr

from util.wkt import rectangular_aoi_select, rectangular_wkt_bounds
from util.util import nisar_start_time_from_url, input_with_default

gdal.UseExceptions()

## Step 2: Initialize All User-Defined Notebook Variables

There are two ways to run this notebook:
- Step through it in JupyterLab, running each code cell.
- Run it as a Python script from the command line using `papermill`.

[papermill](https://github.com/nteract/papermill) is a tool that allows you to parameterize and execute a Jupyter Notebook as a script. This notebook has been configured so that it may be executed manually in JupyterLab by running code cells or from the command line, as a script, using papermill.

:::{hint}
#### papermill command examples:

Run the CalVal data demo from the command line:
```
papermill NISAR_L3_Inundation_ProductGeneration.ipynb NISAR_L3_Inundation_ProductGeneration_papermill_demo_output.ipynb \
-p papermill True \
-p demo True \
-p study_area Yucatan_Lake \
-p combination_method sum
```

Run on GCOV data for a user-defined AOI and time range (This requires that your Earthdata Login credentials are added to a `~/.netrc` file or as environment variables):
```
papermill NISAR_L3_Inundation_ProductGeneration.ipynb NISAR_L3_Inundation_ProductGeneration_papermill_data_search_output.ipynb \
-p papermill True \
-p demo False \
-p study_area Default \
-p combination_method sum \
-p subset_wkt "POLYGON((-91.624689 18.263099, -91.38299 18.263099, -91.38299 18.432551, -91.624689 18.432551, -91.624689 18.263099))" \
-p start_dt "2025-08-01 00:00:00" \
-p end_dt "2026-02-01 23:59:59" \
-p granule_name "*_D_*_DHDH_*"
```
:::

### Step 2a: Initialize Inundation workflow parameters

:::{hint}
Update the arguments below and in Step 2c to define and process your own NISAR GCOV dataset
:::

In [ ]:
papermill = False
demo = True # Set to True to download calval data. Set to False to process your own NISAR GCOV dataset.
study_area = 'Yucatan_Lake' # Yucatan_Lake, NewOrleans, Default
combination_method = 'sum'  # combination method is either product or sum

### Step 2b: (Optional) Draw an AOI on a map to generate Well-Known-Text for spatial subsetting

If you are generating inundation maps from a custom stack of GCOV data, you must define a spatial subset in Step 2c using a Well-Known-Text (WKT) POLYGON. 

The interactive map produced below allows you to generate WKT for an AOI, which you can copy and paste into Step 2c's `subset_wkt` variable. 

If you are running the demo cal-val dataset or already have the WKT for your AOI, you can skip this step.

In [ ]:
if not demo and not papermill:
    aoi_map = rectangular_aoi_select()
    display(aoi_map["map"])

### Step 2c: If processing a custom-defined stack of GCOV data, overwrite the default search and subset variables below

In [ ]:
if not demo and not papermill:
    subset_wkt = "POLYGON((-91.583458 18.290605, -91.438576 18.290605, -91.438576 18.425507, -91.583458 18.425507, -91.583458 18.290605))"
    start_dt = '2025-08-01 00:00:00'
    end_dt = '2025-12-01 23:59:59'
    granule_name = '*_D_*_DHDH_*' # GCOV product ID search filter

<hr>

## Step 3: Access the Data

Download NISAR Ecosystems Inundation demonstration data from its S3 bucket or access NISAR GCOV data with `earthaccess` 

### Step 3a: Search for data

In [ ]:
if demo:
    gcov_url = ('s3://nisar-public-ebd/ATBD/ecosystems/inundation/*/*')
    s3 = s3fs.S3FileSystem(anon=True,endpoint_url='https://s3.us-west-1.wasabisys.com')
    gcov_paths = ['s3://' + k  for k in s3.glob(gcov_url)]
else:
    subset_bounds = rectangular_wkt_bounds(subset_wkt)
    bbox = dict(
        minx=float(subset_bounds["west"]), 
        miny=float(subset_bounds["south"]), 
        maxx=float(subset_bounds["east"]), 
        maxy=float(subset_bounds["north"]), 
        crs="EPSG:4326")
    temporal = (start_dt, end_dt)
    kwargs = {
        'granule_name': granule_name,
        'bounding_box': (bbox["minx"], bbox["miny"], bbox["maxx"], bbox["maxy"]),
        'temporal': temporal,
        'short_name': 'NISAR_L2_GCOV_BETA_V1'
    }
    auth = earthaccess.login()
    results = earthaccess.search_data(**kwargs)
    gcov_paths = [r.data_links(access='external')[0] for r in results]
gcov_paths = sorted(gcov_paths)
print("number of available scenes:", len(gcov_paths))

### Step 3b: Load the data

In [ ]:
%%time

group_path = "/science/LSAR/GCOV/grids/frequencyA" # change this to any GCOV HDF5 group you wish

kwargs = {
    "cache_type": "background",
    "block_size": 8 * 1024 * 1024,  # 8 MB
}

if demo:
    s3_files = [s3.open(url, "rb", **kwargs) for url in gcov_paths]
    datatrees = [
        xr.open_datatree(
            f,
            engine="h5netcdf",
            decode_timedelta=False,
            phony_dims="access",
            chunks="auto",
            group=group_path,
        )
        for f in s3_files
    ]
else:
    fs = earthaccess.get_fsspec_https_session()
    datatrees = [
        xr.open_datatree(
            fs.open(f, **kwargs),
            engine="h5netcdf",
            decode_timedelta=False,
            phony_dims="access",
            chunks="auto",
            group=group_path,
        )
        for f in gcov_paths
    ]

### Step 3c: Create a time series xarray.Dataset

In [ ]:
%%time

datetimes = [nisar_start_time_from_url(gcov_path) for gcov_path in gcov_paths]

dataarrays = [
    tree.ds.assign_coords(time=dt).expand_dims(time=1)[['HHHH', 'HVHV', 'projection', 'xCoordinateSpacing', 'yCoordinateSpacing']]
    for dt, tree in zip(datetimes, datatrees)
]

for i, da in enumerate(dataarrays):
    da = da.rio.write_crs(f"EPSG:{da.projection.item()}")
    if not demo:
        dataarrays[i] = da.rio.clip_box(**bbox)

ts = xr.concat(dataarrays, dim="time")
ts = ts.persist()
ts

### Step 3d: Confirm that all data are projected to the same EPSG

In [ ]:
epsgs = [ts.sel(time=t).rio.crs.to_epsg() if dataarrays[0].projection.item() == 0 else dataarrays[0].projection.item() for t in ts.time]
epsg_equal = len(set(epsgs)) == 1
if epsg_equal:
    print(f"Time series contains single EPSG: {epsgs[0]}")
else:
    raise Exception(f"Time series contains multiple EPSGs: {set(epsgs)}")

num_files = len(ts.time)

<hr>

## Step 4: Plot the Time Series

### Step 4a: Plot the HHHH and HVHV data



In [ ]:
# plot the GCOV images

fig, axs = plt.subplots(num_files, 2, figsize=(12,num_files*5))
cbar_shrink = 0.6

for i, t in enumerate(ts.time):

    im1 = axs[i][0].imshow(ts.sel(time=t).HHHH, vmin=0, vmax=0.5, cmap='gray')
    title_str = f"{str(ts.sel(time=t).time.item().date())}, HHHH"
    axs[i][0].set_title(title_str)
    fig.colorbar(im1, ax=axs[i][0], shrink=cbar_shrink)

    im2 = axs[i][1].imshow(ts.sel(time=t).HVHV, vmin=0, vmax=0.1, cmap='gray')
    title_str = f"{str(ts.sel(time=t).time.item().date())}, HVHV"
    axs[i][1].set_title(title_str)
    fig.colorbar(im2, ax=axs[i][1], shrink=cbar_shrink)

### Step 4b: Generate the ratio, product, and sum of the HHHH and HVHV data

In [ ]:
%%time

# Compute ratio, product, and sum
ts["hhhh_hvhv_ratio"] = ts.HHHH / ts.HVHV
ts["hhhh_hvhv_product"] = ts.HHHH * ts.HVHV
ts["hhhh_hvhv_sum"] = ts.HHHH + ts.HVHV
ts

### Step 4c: Plot the ratio, product, and sum of the HHHH and HVHV data

In [ ]:
# Plot ratio, product, sum
fig, axs = plt.subplots(num_files, 3, figsize=(14,num_files*5))
cbar_shrink = 0.6

for i, t in enumerate(ts.time):

    im1 = axs[i][0].imshow(ts.sel(time=t).hhhh_hvhv_ratio, vmin=1, vmax=12, cmap='gray')
    title_str = f"{str(ts.sel(time=t).time.item().date())}, hhhh_hvhv_ratio"
    axs[i][0].set_title(title_str)
    fig.colorbar(im1, ax=axs[i][0], shrink=cbar_shrink)

    im2 = axs[i][1].imshow(ts.sel(time=t).hhhh_hvhv_product, vmin=0, vmax=0.05, cmap='gray')
    title_str = f"{str(ts.sel(time=t).time.item().date())}, hhhh_hvhv_product"
    axs[i][1].set_title(title_str)
    fig.colorbar(im2, ax=axs[i][1], shrink=cbar_shrink)

    im3 = axs[i][2].imshow(ts.sel(time=t).hhhh_hvhv_sum, vmin=0, vmax=0.5, cmap='gray')
    title_str = f"{str(ts.sel(time=t).time.item().date())}, hhhh_hvhv_sum"
    axs[i][2].set_title(title_str)
    fig.colorbar(im2, ax=axs[i][2], shrink=cbar_shrink)

<hr>

## Step 5: Define the Classification Thresholds

Classification thresholds options are defined in the [wetland_calibration_parameters.yaml config file](../ancillary_data/wetland_calibration_parameters.yaml) 

### Step 5a: Read in all configuration options for your selected configuration site

Looking at the Yucatan Lake site configuration, we see that only `inun_veg_single_class` and `open_water` are used. For `open_water`, the sum is used, not the product

In [ ]:
# Read in contents of yaml configuration file
with open(Path.cwd().parent / 'ancillary_data' / 'wetland_calibration_parameters.yaml','rb') as f:    
    config_doc = safe_load(f)

class_thresh = config_doc['runconfig']['calval_sites'][study_area]
class_thresh

### 5b: Save the configuration options for your selected site using the chosen combination method (sum or product)

In [ ]:
th = {}

# inundated vegetation (iv)
th['iv_hh_max'] = class_thresh['inun_veg_single_class']['HH_max']
th['iv_hh_min'] = class_thresh['inun_veg_single_class']['HH_min']
th['iv_ratio_max'] = class_thresh['inun_veg_single_class']['ratio_max']
th['iv_ratio_min'] = class_thresh['inun_veg_single_class']['ratio_min']

# open water (ow)
if combination_method == 'sum':
    th['ow_comb_max'] = class_thresh['open_water']['sum_max']
    th['ow_comb_min'] = class_thresh['open_water']['sum_min']
elif combination_method == 'product':
    th['ow_comb_max'] = class_thresh['open_water']['product_max']
    th['ow_comb_min'] = class_thresh['open_water']['product_min']    
else:
    raise Exception("Invalid combination method") 

th

<hr>

## Step 6: Classify the GCOV data

In [ ]:
# First classify open water, then classify remaining (non-open water) pixels as inun veg or not inundated

np.seterr(invalid='ignore')

ts["inundation_classified"] = xr.zeros_like(ts.HHHH)
numpx = ts.isel(time=0).HHHH.notnull().sum().to_numpy()

for t in ts.time:
    ts.sel(time=t)["inundation_classified"] = xr.zeros_like(ts.sel(time=t).HHHH)
    np.zeros(ts.sel(time=t).HHHH.shape, dtype=np.int8)


    # set all valid data pixels to 1
    if combination_method == 'sum':
        ds_comb = ts.sel(time=t).hhhh_hvhv_sum.copy()
    else:
        ds_comb = ts.sel(time=t).hhhh_hvhv_product.copy()
    idx = ds_comb > 0
    ts["inundation_classified"].loc[dict(time=t)] = xr.where(
        idx,
        1,
        ts.sel(time=t)["inundation_classified"]
    )

    # set open water pixels to 2
    idx = (ds_comb > th['ow_comb_min']) & (ds_comb <= th['ow_comb_max'])
    ts["inundation_classified"].loc[dict(time=t)] = xr.where(
        idx,
        2,
        ts.sel(time=t)["inundation_classified"]
    )

    # set inundate vegetation pixels to 3
    idx = (ts.sel(time=t).HHHH >= th['iv_hh_min']) & (ts.sel(time=t).HHHH <= th['iv_hh_max']) & \
           (ts.sel(time=t).hhhh_hvhv_ratio >= th['iv_ratio_min']) & (ts.sel(time=t).hhhh_hvhv_ratio <= th['iv_ratio_max']) & \
           (ts.sel(time=t)["inundation_classified"] != 2)
    ts["inundation_classified"].loc[dict(time=t)] = xr.where(
        idx,
        3,
        ts.sel(time=t)["inundation_classified"]
    )

    print(str(ts.time.sel(time=t).item().date()))
    print(f"Open Water: {(100*(ts.sel(time=t)["inundation_classified"] == 2).sum().to_numpy()/numpx):.2f}%")
    print(f"Inundated Vegetation: {(100*(ts.sel(time=t)["inundation_classified"] == 3).sum().to_numpy()/numpx):.2f}%")
    print(f"Not Inundated: {(100*(ts.sel(time=t)["inundation_classified"] == 1).sum().to_numpy()/numpx):.2f}%\n")


<hr>

## Step 7: Plot the Classified Images at Native Resolution

In [ ]:
# plot classified images

# set up colormaps
c_white = (255, 255, 255)
c_lightblue = (66, 233, 245)
c_darkblue = (21, 27, 115)
c_gray = (236, 236, 238)

colors = [c_white, c_gray, c_darkblue, c_lightblue]
colors2 = []
for k in colors:
    colors2.append(tuple(np.array(k)/255)) 
cmap = LinearSegmentedColormap.from_list('cmap_class', colors2, N=4)

fig, axs = plt.subplots(num_files, 1, figsize=(6,num_files*5))
cbar_shrink = 0.6
cbar_ticks = [3/8, 9/8, 15/8, 21/8]
cbar_labels = ['no data','not inun','open water','inun veg']
# cbar_ticks = [4/10, 12/10, 20/10, 28/10, 36/10]  # for 2 inun veg classes
# cbar_label = ['no data','not inun','open water','inun veg I','inun veg II']  # for 2 inun veg classes

for i, t in enumerate(ts.time):
    im = axs[i].imshow(ts.sel(time=t)["inundation_classified"], vmin=0, vmax=3, cmap=cmap, interpolation='nearest')
    resolution = int(ts.sel(time=t).xCoordinateSpacing.item())
    axs[i].set_title(f"{str(ts.sel(time=t).time.item().date())} inundation_classified {resolution}m")
    cbar = plt.colorbar(im, ax=axs[i], shrink=cbar_shrink)    
    cbar.set_ticks(cbar_ticks)
    cbar.set_ticklabels(cbar_labels, fontsize=10)  

<hr>

## Step 8: Aggregate to 1 Hectare and Export as GeoTiff

In [ ]:
# output results as geotiff using rasterio

class_dir = Path().cwd() / 'nisar_classifications' / study_area
Path(class_dir).mkdir(parents=True, exist_ok=True)

ts[f"classification_{resolution}m_filepath"] = (
    "time",
    np.full(ts.sizes["time"], "", dtype="U512")
)

ts["classification_1ha_filepath"] = (
    "time",
    np.full(ts.sizes["time"], "", dtype="U512")
)

for i, t in enumerate(ts.time):

    # create native resolution geotiffs to use as inputs for gdal Warp
    filepath_native_res = str(class_dir / (f'nisar_classified_{resolution}m_' + datetime.today().strftime('%Y%m%d') + '_gcov_' + str(ts.sel(time=t).time.item().date()) + \
                      '_' + study_area + '.tif'))
    ts[f"classification_{resolution}m_filepath"].loc[{"time": t}] = filepath_native_res
    meta = {'driver': 'GTiff', 
            'dtype': 'float32', 
            'nodata': None, 
            'width': ts.sizes["xCoordinates"], 
            'height': ts.sizes["yCoordinates"], 
            'count': 1, 
            'crs': ts.rio.crs, 
            'transform': ts.rio.transform(),
            'tiled': False, 
            'interleave': 'band'}
    with rasterio.open(filepath_native_res, 'w', **meta) as dst:
        dst.write(ts.sel(time=t)["inundation_classified"], indexes=1)    

    L3_filepath_1ha = str(class_dir / ('nisar_classified_1ha_' + datetime.today().strftime('%Y%m%d') + '_gcov_' + str(ts.sel(time=t).time.item().date()) + \
                      '_' + study_area + '.tif'))
    ts["classification_1ha_filepath"].loc[{"time": t}] = L3_filepath_1ha
    gdal.Warp(L3_filepath_1ha, filepath_native_res, xRes=100, yRes=-100, resampleAlg=gdal.GRA_Mode, format="COG")

    # optional: remove native reolution GeoTiffs
    # Path(filepath_native_res).unlink()

<hr>

## Step 9: Plot the Classified Images at 1 Hectare Resolution

In [ ]:
# read in the 1 ha GeoTiffs
nisar_classified_1ha = []

for t in ts.time:
    ds = rioxarray.open_rasterio(ts.sel(time=t)["classification_1ha_filepath"].item())
    nisar_classified_1ha.append(ds.to_numpy().squeeze().astype(np.int8))


# plot 1-ha classified images

# set up colormaps
c_white = (255, 255, 255)
c_lightblue = (66, 233, 245)
c_darkblue = (21, 27, 115)
c_gray = (236, 236, 238)

colors = [c_white, c_gray, c_darkblue, c_lightblue]
colors2 = []
for k in colors:
    colors2.append(tuple(np.array(k)/255)) 
cmap = LinearSegmentedColormap.from_list('cmap_class', colors2, N=4)

fig, axs = plt.subplots(num_files, 1, figsize=(6,num_files*5))
cbar_shrink = 0.6
cbar_ticks = [3/8, 9/8, 15/8, 21/8]
cbar_labels = ['no data','not inun','open water','inun veg']

for i, t in enumerate(ts.time):
    im = axs[i].imshow(nisar_classified_1ha[i], vmin=0, vmax=3, cmap=cmap, interpolation='nearest')
    axs[i].set_title(f"{str(ts.sel(time=t).time.item().date())} 1ha_inundation_classified")
    cbar = plt.colorbar(im, ax=axs[i], shrink=cbar_shrink)    
    cbar.set_ticks(cbar_ticks)
    cbar.set_ticklabels(cbar_labels, fontsize=10)    